In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
from xgboost import XGBRegressor
from scipy.optimize import minimize, least_squares
import statsmodels.api as sm
from tqdm import tqdm
from scipy import stats
from types import SimpleNamespace
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CHARGEMENT ET PRÉPARATION DES DONNÉES
# ============================================================
print("="*80)
print("CHARGEMENT DES DONNÉES")
print("="*80)
data = pd.read_csv("C:/Users/P279390/Documents/Production et Prix en France/historique_2015_2024.csv", parse_dates=["Date"])
df = data.copy()
df = df.sort_values("Date").reset_index(drop=True)
df = df.set_index('Date')
print(f"✓ Données chargées: {df.shape}")

# Suppression des doublons temporels
dupes = df.index.duplicated(keep=False)
if dupes.any():
    print(f"⚠️ {dupes.sum()} doublons trouvés — suppression.")
    df = df[~df.index.duplicated(keep='first')]

# Vérification de la continuité temporelle
expected_index = pd.date_range(df.index.min(), df.index.max(), freq='H')
missing = expected_index.difference(df.index)
print(f"→ {len(missing)} heures manquantes détectées")

if len(missing) > 0:
    print("⚠️ Reconstruction de la série horaire continue (interpolation)")
    df = df.reindex(expected_index)
    for col in ['Prix', 'ChargePrévisions', 'Nucléaire', 'Renouvelable',
                'Non_Renouvelable', 'Exportations', 'Importations', 'Pompage']:
        if col in df.columns:
            df[col] = df[col].interpolate(method='time', limit_direction='both')
    print(f"✓ Série réindexée : {df.shape}")
else:
    print("✅ Série horaire continue.")

# ============================================================
# GESTION DES PRIX EXTRÊMES (> 1000 €/MWh)
# ============================================================
print("\n" + "="*80)
print("GESTION DES PRIX EXTRÊMES")
print("="*80)

prix_extremes_mask = df['Prix'] > 1000
nb_prix_extremes = prix_extremes_mask.sum()

if nb_prix_extremes > 0:
    print(f"⚠️ {nb_prix_extremes} prix > 1000 €/MWh détectés")
    print("   → Interpolation par médiane mobile sur 24h")
    df.loc[prix_extremes_mask, 'Prix'] = df['Prix'].rolling(window=24, center=True, min_periods=6).median()
    print(f"✓ Après correction: { (df['Prix'] > 1000).sum() } prix > 1000 €/MWh")
else:
    print("✓ Aucun prix extrême détecté")

# ============================================================
# AJOUT DES VARIABLES TEMPORELLES ET SAISONNIÈRES
# ============================================================
print("\n" + "="*80)
print("AJOUT DES VARIABLES TEMPORELLES ET SAISONNIÈRES")
print("="*80)

df['Heure'] = df.index.hour
df['JourSemaine'] = df.index.dayofweek
df['Weekend'] = df['JourSemaine'].apply(lambda x: 1 if x >= 5 else 0)

df['sin_hour'] = np.sin(2 * np.pi * df['Heure'] / 24)
df['cos_hour'] = np.cos(2 * np.pi * df['Heure'] / 24)

df['JourAnnee'] = df.index.dayofyear
df['sin_day'] = np.sin(2 * np.pi * df['JourAnnee'] / 365.25)
df['cos_day'] = np.cos(2 * np.pi * df['JourAnnee'] / 365.25)

CALENDAR_VARS = ['sin_hour', 'cos_hour', 'sin_day', 'cos_day', 'JourSemaine', 'Weekend']
print(f"✅ Variables saisonnières ajoutées: {CALENDAR_VARS}")

# ============================================================
# CRÉATION DES LAGS DE PRIX
# ============================================================
print("\n" + "="*80)
print("CRÉATION DES LAGS DE PRIX")
print("="*80)

# Prix_1h: prix à 23h du jour précédent (pas de look-ahead bias)
idx_23 = df[df.index.hour == 23].index
prix_23 = df.loc[idx_23, 'Prix']
map_23 = {d.date(): prix_23.loc[d] for d in prix_23.index}

df['Prix_1h'] = np.nan
for ts in df.index:
    prev_day = (ts - pd.Timedelta(days=1)).date()
    df.at[ts, 'Prix_1h'] = map_23.get(prev_day, np.nan)

# Autres lags
for lag in [24, 48, 168]:
    df[f'Prix_{lag}h'] = df['Prix'].shift(lag)

print("✅ Lags créés:")
print("   - Prix_1h   : prix à 23h du jour précédent")
print("   - Prix_24h  : lag de 24 heures")
print("   - Prix_48h  : lag de 48 heures")
print("   - Prix_168h : lag de 168 heures (1 semaine)")

# ============================================================
# DÉSENDOGÉNÉISATION PAR XGBOOST
# ============================================================
print("\n" + "="*80)
print("DÉSENDOGÉNÉISATION PAR XGBOOST")
print("="*80)

ENDOG = ['Nucléaire', 'Renouvelable', 'Non_Renouvelable',
         'Exportations', 'Importations', 'Pompage']

BASE_LAG_VARS = []
for lag in [1, 24, 48, 168]:
    for var in ENDOG:
        lag_col = f'{var}_lag{lag}'
        df[lag_col] = df[var].shift(lag)
        BASE_LAG_VARS.append(lag_col)

XGB_EXOG_VARS = CALENDAR_VARS + BASE_LAG_VARS

df_clean = df.dropna(subset=BASE_LAG_VARS)

results_xgb = {}
split_date = pd.Timestamp("2024-01-01 00:00:00")

for var in tqdm(ENDOG, desc="Instrumentation XGBoost"):
    X = df_clean[XGB_EXOG_VARS]
    y = df_clean[var]

    X_train = X[X.index < split_date]
    y_train = y[y.index < split_date]
    X_test = X[X.index >= split_date]
    y_test = y[y.index >= split_date]

    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        reg_alpha=0.5,
        min_child_weight=5,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train, verbose=False)

    y_hat = model.predict(X)
    df_clean[f'{var}_hat'] = y_hat

    r2_train = r2_score(y_train, model.predict(X_train))
    r2_test = r2_score(y_test, model.predict(X_test))
    results_xgb[var] = {'R²_train': r2_train, 'R²_test': r2_test}

    print(f"  ✓ {var:20s} : R²_train={r2_train:.3f}, R²_test={r2_test:.3f}")

for var in ENDOG:
    df[f'{var}_hat'] = df_clean[f'{var}_hat']

print("✅ Désendogénéisation terminée")

# ============================================================
# CONSTRUCTION DE X_t
# ============================================================
print("\n" + "="*80)
print("CONSTRUCTION DE X_t")
print("="*80)

df['xt'] = (df["Renouvelable_hat"] + df["Nucléaire_hat"] + df['Importations_hat']) \
             - (df["ChargePrévisions"] + df['Exportations_hat'] + df['Pompage_hat'])

print(f"✓ x_t = Bilan offre-demande instrumenté")
print(f"  NaN dans x_t: {df['xt'].isna().sum()}")

df = df.dropna(subset=['xt', 'Prix', 'Prix_1h', 'Prix_24h', 'Prix_48h', 'Prix_168h'])
print(f"✅ Données nettoyées : {df.shape}")

# ============================================================
# SPLIT TRAIN/VALIDATION/TEST
# ============================================================
print("\n" + "="*80)
print("SPLIT TRAIN/VALIDATION/TEST")
print("="*80)

split_date_val = "2023-07-01"
split_date_test = "2024-01-01"

df_train = df.loc[df.index < split_date_val].copy()
df_val = df.loc[(df.index >= split_date_val) & (df.index < split_date_test)].copy()
df_test = df.loc[df.index >= split_date_test].copy()

print(f"✓ Train:      {df_train.shape[0]:>6} obs  "
      f"({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"✓ Validation: {df_val.shape[0]:>6} obs  "
      f"({df_val.index[0].date()} → {df_val.index[-1].date()})")
print(f"✓ Test:       {df_test.shape[0]:>6} obs  "
      f"({df_test.index[0].date()} → {df_test.index[-1].date()})")

# Vérification des prix négatifs par split
print(f"\n→ Taux de prix négatifs par split:")
print(f"   • Train:      {(df_train['Prix'] < 0).mean():.3%}  "
      f"({(df_train['Prix'] < 0).sum()} heures)")
print(f"   • Validation: {(df_val['Prix'] < 0).mean():.3%}  "
      f"({(df_val['Prix'] < 0).sum()} heures) ← doit être suffisant pour calibrer")
print(f"   • Test:       {(df_test['Prix'] < 0).mean():.3%}  "
      f"({(df_test['Prix'] < 0).sum()} heures)")

# ============================================================
# ANALYSE STATISTIQUE DE xt (TRAIN)
# ============================================================
def analyse_distribution(name, s):
    s_nonan = s.dropna()
    print(f"=== Distribution de {name} (train) ===")
    print(f"min           : {s_nonan.min():.4f}")
    print(f"1er quartile  : {np.percentile(s_nonan,25):.4f}")
    print(f"médiane       : {np.median(s_nonan):.4f}")
    print(f"3e quartile   : {np.percentile(s_nonan,75):.4f}")
    print(f"max           : {s_nonan.max():.4f}")
    print(f"IQR           : {np.percentile(s_nonan,75) - np.percentile(s_nonan,25):.4f}")
    print("\n")

analyse_distribution("xt", df_train["xt"])

# ============================================================
# NORMALISATION ROBUSTE DE X_t
# ============================================================
scaler_xt = RobustScaler()

df_train.loc[:, ['xt']] = scaler_xt.fit_transform(df_train[['xt']])
df_val.loc[:, ['xt']] = scaler_xt.transform(df_val[['xt']])
df_test.loc[:, ['xt']] = scaler_xt.transform(df_test[['xt']])

df.loc[df_train.index, 'xt'] = df_train['xt']
df.loc[df_val.index, 'xt'] = df_val['xt']
df.loc[df_test.index, 'xt'] = df_test['xt']

analyse_distribution("xt", df_train["xt"])

# ============================================================
# TRANSFORMATION ASINH DES PRIX ET LAGS
# ============================================================
def robust_params(x):
    a = np.median(x)
    mad = np.median(np.abs(x - a))
    b = 1.4826 * mad
    return a, (b if b != 0 else 1.0)

def robust_normalize(x, a, b):
    return (x - a) / b

def asinh_transform(x):
    return np.arcsinh(x)

def asinh_inverse(x):
    return np.sinh(x)

def retransform_price(y_trans, a_train, b_train):
    y_norm = asinh_inverse(y_trans)
    y_orig = y_norm * b_train + a_train
    return y_orig

# Vérification de la transformation
test_val = np.array([-2, -1, 0, 1, 2])
test_trans = asinh_transform(test_val)
test_back = asinh_inverse(test_trans)
assert np.allclose(test_val, test_back, atol=1e-10), "Transformation asinh/sinh non inversible"
print("✓ Transformation asinh/sinh vérifiée (inversible)")

a_train, b_train = robust_params(df_train['Prix'])
print(f"✓ Paramètres robustes (Prix train): médiane={a_train:.2f}, scale={b_train:.2f}")

prix_vars = ['Prix', 'Prix_1h', 'Prix_24h', 'Prix_48h', 'Prix_168h']
for var in prix_vars:
    df_train.loc[:, f'{var}_norm'] = robust_normalize(df_train[var], a_train, b_train)
    df_val.loc[:, f'{var}_norm'] = robust_normalize(df_val[var], a_train, b_train)
    df_test.loc[:, f'{var}_norm'] = robust_normalize(df_test[var], a_train, b_train)
    
    df_train.loc[:, f'{var}_trans'] = asinh_transform(df_train[f'{var}_norm'])
    df_val.loc[:, f'{var}_trans'] = asinh_transform(df_val[f'{var}_norm'])
    df_test.loc[:, f'{var}_trans'] = asinh_transform(df_test[f'{var}_norm'])
    
    df.loc[df_train.index, f'{var}_trans'] = df_train[f'{var}_trans']
    df.loc[df_val.index, f'{var}_trans'] = df_val[f'{var}_trans']
    df.loc[df_test.index, f'{var}_trans'] = df_test[f'{var}_trans']

# ============================================================
# MODÈLE AVEC CONTRAINTES MINIMALES
# ============================================================
print("\n" + "="*80)
print("🏆 MODÈLE AVEC CONTRAINTES MINIMALES")
print("="*80)

def model_optimal_minimal(params, xt_arr, y_lags_arr, residuals_lag1=None):
    """
    MODÈLE AVEC CONTRAINTES MINIMALES:
    - Tous les α doivent être positifs
    - θ₁ et θ₂₄ sont contraints à |θ| < 1
    """
    # INTERCEPT - LIBRE mais α > 0
    phi_0 = params[0]
    theta_0 = params[1]
    alpha_0 = max(params[2], 0.01)  # α > 0.01
    
    # LAG 1h - CONTRAINT
    phi_1 = params[3]
    theta_1 = np.clip(params[4], -0.99, 0.99)  # |θ| < 1
    alpha_1 = max(params[5], 0.01)
    
    # LAG 24h - CONTRAINT
    phi_24 = params[6]
    theta_24 = np.clip(params[7], -0.99, 0.99)  # |θ| < 1
    alpha_24 = max(params[8], 0.01)
    
    # LAG 48h - LIBRE mais α > 0
    phi_48 = params[9]
    theta_48 = params[10]
    alpha_48 = max(params[11], 0.01)
    
    # LAG 168h - LIBRE mais α > 0
    phi_168 = params[12]
    theta_168 = params[13]
    alpha_168 = max(params[14], 0.01)
    
    intercept = phi_0 + theta_0 * np.exp(alpha_0 * xt_arr)
    lag_1 = (phi_1 + theta_1 * np.exp(alpha_1 * xt_arr)) * y_lags_arr[:, 0]
    lag_24 = (phi_24 + theta_24 * np.exp(alpha_24 * xt_arr)) * y_lags_arr[:, 1]
    lag_48 = (phi_48 + theta_48 * np.exp(alpha_48 * xt_arr)) * y_lags_arr[:, 2]
    lag_168 = (phi_168 + theta_168 * np.exp(alpha_168 * xt_arr)) * y_lags_arr[:, 3]
    
    y_pred = intercept + lag_1 + lag_24 + lag_48 + lag_168
    
    if residuals_lag1 is not None and len(params) > 15:
        phi_ar = np.clip(params[15], -0.5, 0.5)
        y_pred = y_pred + phi_ar * residuals_lag1
    
    return y_pred

print("✅ Modèle avec contraintes minimales défini")

# ============================================================
# INITIALISATION ROBUSTE PAR LINÉARISATION (OLS)
# ============================================================
print("\n" + "="*80)
print("INITIALISATION ROBUSTE PAR LINÉARISATION (OLS)")
print("="*80)

df_train_val = pd.concat([df_train, df_val])
y_init = df_train_val['Prix_trans'].values

X_init = df_train_val[['xt', 'Prix_1h_trans', 'Prix_24h_trans', 'Prix_48h_trans', 'Prix_168h_trans']].copy()
X_init['xt*Prix_1h'] = X_init['xt'] * X_init['Prix_1h_trans']
X_init['xt*Prix_24h'] = X_init['xt'] * X_init['Prix_24h_trans']
X_init['xt*Prix_48h'] = X_init['xt'] * X_init['Prix_48h_trans']
X_init['xt*Prix_168h'] = X_init['xt'] * X_init['Prix_168h_trans']

X_init = sm.add_constant(X_init, prepend=True)
ols_init = sm.OLS(y_init, X_init, missing='drop').fit()
print("✓ OLS sur ensemble train+val terminé")

coefs = ols_init.params

delta_int = float(coefs.get('const', 0.0))
delta_x = float(coefs.get('xt', 0.0))

phi_1_init = float(coefs.get('Prix_1h_trans', 0.0))
phi_24_init = float(coefs.get('Prix_24h_trans', 0.0))
phi_48_init = float(coefs.get('Prix_48h_trans', 0.0))
phi_168_init = float(coefs.get('Prix_168h_trans', 0.0))

theta_1_init = float(coefs.get('xt*Prix_1h', 0.0))
theta_24_init = float(coefs.get('xt*Prix_24h', 0.0))
theta_48_init = float(coefs.get('xt*Prix_48h', 0.0))
theta_168_init = float(coefs.get('xt*Prix_168h', 0.0))

iqr_xt = np.percentile(df_train["xt"].dropna(), 75) - np.percentile(df_train["xt"].dropna(), 25)
alpha_init = 0.5 * iqr_xt if iqr_xt != 0 else 0.1

theta_0_init = delta_x / alpha_init
phi_0_init = delta_int - theta_0_init

phi_ar_init = 0.1

initial_params = np.array([
    phi_0_init, theta_0_init, alpha_init,
    phi_1_init, theta_1_init, alpha_init,
    phi_24_init, theta_24_init, alpha_init,
    phi_48_init, theta_48_init, alpha_init,
    phi_168_init, theta_168_init, alpha_init,
    phi_ar_init
], dtype=float)

param_names = [
    'phi_0', 'theta_0', 'alpha_0',
    'phi_1', 'theta_1', 'alpha_1',
    'phi_24', 'theta_24', 'alpha_24',
    'phi_48', 'theta_48', 'alpha_48',
    'phi_168', 'theta_168', 'alpha_168',
    'phi_ar'
]

print("\n✅ Initialisation robuste (16 paramètres):")
for i, (name, val) in enumerate(zip(param_names, initial_params)):
    print(f"  {i+1:2d}. {name:8s} = {val:.6f}")

# ============================================================
# OPTIMISATION AVEC VALIDATION CROISÉE
# ============================================================
print("\n" + "="*80)
print("OPTIMISATION AVEC VALIDATION CROISÉE")
print("="*80)

y_train = df_train['Prix_trans'].values
y_val = df_val['Prix_trans'].values

y_lags_train = df_train[['Prix_1h_trans', 'Prix_24h_trans', 'Prix_48h_trans', 'Prix_168h_trans']].values
y_lags_val = df_val[['Prix_1h_trans', 'Prix_24h_trans', 'Prix_48h_trans', 'Prix_168h_trans']].values

xt_train = df_train['xt'].values
xt_val = df_val['xt'].values

print(f"\nTrain: {len(y_train)} observations, Validation: {len(y_val)} observations")

def objective_minimal(params):
    y_hat = model_optimal_minimal(params, xt_train, y_lags_train)
    return float(np.mean((y_train - y_hat)**2))

def residuals_func_minimal(params, xt_data, y_lags_data, y_obs):
    y_pred = model_optimal_minimal(params, xt_data, y_lags_data)
    return (y_obs - y_pred).ravel()

# Optimisation L-BFGS-B
print("\n→ Optimisation L-BFGS-B")
res_opt = minimize(objective_minimal, initial_params, method='L-BFGS-B',
                   options={'maxiter': 5000, 'ftol': 1e-9, 'gtol': 1e-7})

mse_train = objective_minimal(res_opt.x)
mse_val = np.mean((y_val - model_optimal_minimal(res_opt.x, xt_val, y_lags_val))**2)
print(f"  ✓ MSE train: {mse_train:.6e}")
print(f"  ✓ MSE validation: {mse_val:.6e}")

# Affinement least_squares
print("\n→ Affinement least_squares")
res_ls = least_squares(residuals_func_minimal, res_opt.x,
                       args=(xt_train, y_lags_train, y_train),
                       method='trf', ftol=1e-10, xtol=1e-10, gtol=1e-10,
                       max_nfev=5000, verbose=0)

mse_train_ls = objective_minimal(res_ls.x)
mse_val_ls = np.mean((y_val - model_optimal_minimal(res_ls.x, xt_val, y_lags_val))**2)
print(f"  ✓ MSE train final: {mse_train_ls:.6e}")
print(f"  ✓ MSE validation final: {mse_val_ls:.6e}")

# Sélection du meilleur modèle sur validation
if mse_val_ls < mse_val:
    params_hat = res_ls.x
    best_method = "least_squares"
    best_mse_val = mse_val_ls
else:
    params_hat = res_opt.x
    best_method = "L-BFGS-B"
    best_mse_val = mse_val

print(f"\n✅ Meilleur modèle: {best_method} (MSE val = {best_mse_val:.6e})")

# ============================================================
# RÉSIDUS POUR BOOTSTRAP
# ============================================================
y_pred_train = model_optimal_minimal(params_hat[:15], xt_train, y_lags_train)
residuals_train = y_train - y_pred_train
residuals_train_centered = residuals_train - np.mean(residuals_train)

print(f"\n✓ Résidus entraînement: moyenne={np.mean(residuals_train):.6e}, σ={np.std(residuals_train):.6f}")

# ============================================================
# BOOTSTRAP RÉCURSIF
# ============================================================
print("\n" + "="*80)
print("BOOTSTRAP RÉCURSIF")
print("="*80)

B = 199
warmup = 168  # 7 jours * 24h (période de chauffe pour les lags)
T = len(y_train)

bootstrap_params = []
bootstrap_success = 0

print(f"\nBootstrap récursif B={B} (warmup={warmup}h)")

for b in tqdm(range(B)):
    try:
        eps_star = np.random.choice(residuals_train_centered, size=(T - warmup), replace=True)

        y_star = np.empty(T)
        y_star[:warmup] = y_train[:warmup]

        for t in range(warmup, T):
            p1 = y_star[t-1]
            p24 = y_star[t-24]
            p48 = y_star[t-48]
            p168 = y_star[t-168]

            yhat_t = model_optimal_minimal(params_hat[:15], xt_train[t:t+1],
                                          np.array([[p1, p24, p48, p168]]))[0]
            y_star[t] = yhat_t + eps_star[t-warmup]

        y_lags_star = np.column_stack([
            y_star[warmup-1:T-1],
            y_star[warmup-24:T-24],
            y_star[warmup-48:T-48],
            y_star[warmup-168:T-168]
        ])

        xt_star = xt_train[warmup:]
        y_star_est = y_star[warmup:]

        res_b = least_squares(
            residuals_func_minimal,
            x0=params_hat,
            args=(xt_star, y_lags_star, y_star_est),
            method='trf', ftol=1e-8, xtol=1e-8, gtol=1e-8,
            max_nfev=3000, verbose=0
        )

        if res_b.success:
            bootstrap_params.append(res_b.x.copy())
            bootstrap_success += 1

    except Exception as e:
        if b % 50 == 0:
            print(f"⚠️ Erreur bootstrap rép {b}: {e}")
        continue

bootstrap_params = np.array(bootstrap_params)
print(f"\nBootstrap terminé: {bootstrap_success}/{B} répliques réussies.")

# ============================================================
# ANALYSE DE STABILITÉ DES PARAMÈTRES
# ============================================================
print("\n" + "="*80)
print("ANALYSE DE STABILITÉ DES PARAMÈTRES")
print("="*80)

if bootstrap_success >= 50:
    bootstrap_mean = np.mean(bootstrap_params, axis=0)
    bootstrap_std = np.std(bootstrap_params, axis=0, ddof=1)
    cv = bootstrap_std / np.abs(bootstrap_mean)
    
    print("\n📊 Stabilité des paramètres (CV = écart-type / moyenne):")
    for i, name in enumerate(param_names[:15]):
        print(f"  {name:10s}: estimation={params_hat[i]:.4f}, CV={cv[i]:.4f}")
        if cv[i] > 0.5:
            print(f"    ⚠️  Instable (CV > 0.5)")
else:
    print(f"⚠️ Bootstrap insuffisant ({bootstrap_success}/199)")

# ============================================================
# CALCUL DES SEUILS POUR CHAQUE COMPOSANTE
# ============================================================
print("\n" + "="*80)
print("CALCUL DES SEUILS POUR LE MODÈLE")
print("="*80)

# Récupération des paramètres
phi0, theta0, alpha0 = params_hat[0:3]
phi1, theta1, alpha1 = params_hat[3:6]
phi24, theta24, alpha24 = params_hat[6:9]
phi48, theta48, alpha48 = params_hat[9:12]
phi168, theta168, alpha168 = params_hat[12:15]

# Seuils pour chaque composante (calcul conditionnel)
tau_hat_intercept = -np.log(np.abs(theta0)) / alpha0 if (alpha0 != 0 and theta0 != 0) else np.nan
tau_hat_lag1 = -np.log(np.abs(theta1)) / alpha1 if (alpha1 != 0 and theta1 != 0) else np.nan
tau_hat_lag24 = -np.log(np.abs(theta24)) / alpha24 if (alpha24 != 0 and theta24 != 0) else np.nan
tau_hat_lag48 = -np.log(np.abs(theta48)) / alpha48 if (alpha48 != 0 and theta48 != 0) else np.nan
tau_hat_lag168 = -np.log(np.abs(theta168)) / alpha168 if (alpha168 != 0 and theta168 != 0) else np.nan

print("→ Seuils (espace normalisé) :")
print(f"  • tau_intercept = {tau_hat_intercept:.6f}")
print(f"  • tau_lag1      = {tau_hat_lag1:.6f}")
print(f"  • tau_lag24     = {tau_hat_lag24:.6f}")
print(f"  • tau_lag48     = {tau_hat_lag48:.6f}")
print(f"  • tau_lag168    = {tau_hat_lag168:.6f}")

# Seuils à l'échelle originale
median_x = scaler_xt.center_[0]
iqr_x = scaler_xt.scale_[0]

tau_orig_intercept = median_x + iqr_x * tau_hat_intercept if not np.isnan(tau_hat_intercept) else np.nan
tau_orig_lag1 = median_x + iqr_x * tau_hat_lag1 if not np.isnan(tau_hat_lag1) else np.nan
tau_orig_lag24 = median_x + iqr_x * tau_hat_lag24 if not np.isnan(tau_hat_lag24) else np.nan
tau_orig_lag48 = median_x + iqr_x * tau_hat_lag48 if not np.isnan(tau_hat_lag48) else np.nan
tau_orig_lag168 = median_x + iqr_x * tau_hat_lag168 if not np.isnan(tau_hat_lag168) else np.nan

print(f"\n→ Seuils à l'échelle originale :")
print(f"  • τ_intercept = {tau_orig_intercept:.0f} MWh")
print(f"  • τ_lag1      = {tau_orig_lag1:.0f} MWh")
print(f"  • τ_lag24     = {tau_orig_lag24:.0f} MWh")
print(f"  • τ_lag48     = {tau_orig_lag48:.0f} MWh")
print(f"  • τ_lag168    = {tau_orig_lag168:.0f} MWh")

# Bootstrap pour les seuils (avec vérification)
theta_b = bootstrap_params[:, [1, 4, 7, 10, 13]]  # tous les theta
alpha_b = bootstrap_params[:, [2, 5, 8, 11, 14]]  # tous les alpha

# Calcul conditionnel pour éviter les divisions par zéro
tau_b_matrix = np.full(theta_b.shape, np.nan)
for i in range(theta_b.shape[1]):
    for j in range(theta_b.shape[0]):
        if alpha_b[j, i] != 0 and theta_b[j, i] != 0:
            tau_b_matrix[j, i] = -np.log(np.abs(theta_b[j, i])) / alpha_b[j, i]

# Filtrer les valeurs valides
valid_taus = []
for i in range(tau_b_matrix.shape[1]):
    tau_b_col = tau_b_matrix[:, i]
    tau_b_valid = tau_b_col[np.isfinite(tau_b_col)]
    if len(tau_b_valid) < len(tau_b_col):
        print(f"⚠️ {len(tau_b_col) - len(tau_b_valid)} valeurs invalides pour tau_{i}")
    valid_taus.append(tau_b_valid)

# Calculer les IC pour chaque seuil
tau_hats = [tau_hat_intercept, tau_hat_lag1, tau_hat_lag24, tau_hat_lag48, tau_hat_lag168]
tau_names = ['Intercept', 'Lag1h', 'Lag24h', 'Lag48h', 'Lag168h']

print(f"\n→ Intervalles de confiance (95%) pour les seuils (normalisés):")
for i, (name, tau_hat, tau_b) in enumerate(zip(tau_names, tau_hats, valid_taus)):
    if len(tau_b) > 0:
        ci = np.percentile(tau_b, [2.5, 97.5])
        print(f"  • {name:10s}: {tau_hat:.3f}, CI = [{ci[0]:.3f}, {ci[1]:.3f}]")
    else:
        print(f"  • {name:10s}: Pas de valeurs bootstrap valides")

# ============================================================
# AFFICHAGE FORMATÉ DES RÉSULTATS
# ============================================================
print("\n" + "="*120)
print("RÉSULTATS DU MODÈLE")
print("="*120)

if bootstrap_success < 50:
    print(f"⚠️ Attention: seulement {bootstrap_success} répliques bootstrap réussies")
    print("  Les intervalles de confiance peuvent être peu fiables")
else:
    bootstrap_mean = np.mean(bootstrap_params, axis=0)
    bootstrap_std = np.std(bootstrap_params, axis=0, ddof=1)

    t_stats = params_hat / bootstrap_std
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=bootstrap_success-1))
    significant = p_values < 0.05

    results_params = pd.DataFrame({
        'Groupe': ['Intercept', 'Intercept', 'Intercept',
                   'Lag 1h', 'Lag 1h', 'Lag 1h',
                   'Lag 24h', 'Lag 24h', 'Lag 24h',
                   'Lag 48h', 'Lag 48h', 'Lag 48h',
                   'Lag 168h', 'Lag 168h', 'Lag 168h'],
        'Paramètre': param_names[:15],
        'Estimation': params_hat[:15],
        'Bootstrap Mean': bootstrap_mean[:15],
        'Écart-type': bootstrap_std[:15],
        'IC 95% - Inf': np.percentile(bootstrap_params[:, :15], 2.5, axis=0),
        'IC 95% - Sup': np.percentile(bootstrap_params[:, :15], 97.5, axis=0),
        't-stat': t_stats[:15],
        'p-value': p_values[:15],
        'Significatif (p<0.05)': significant[:15]
    })

    print("\n" + "-"*120)
    print("PARAMÈTRES ESTIMÉS")
    print("-"*120)
    print(results_params.to_string(index=False, float_format=lambda x: f'{x:.6f}' if abs(x) < 100 else f'{x:.3f}'))

# ============================================================
# ANALYSE DES EFFETS NON-LINÉAIRES PAR COMPOSANTE
# ============================================================
print("\n" + "="*80)
print("ANALYSE DES EFFETS NON-LINÉAIRES PAR COMPOSANTE")
print("="*80)

xt_range = np.linspace(df_train['xt'].min(), df_train['xt'].max(), 100)

effects = {
    'Intercept': params_hat[0] + params_hat[1] * np.exp(params_hat[2] * xt_range),
    'Lag 1h': params_hat[3] + params_hat[4] * np.exp(params_hat[5] * xt_range),
    'Lag 24h': params_hat[6] + params_hat[7] * np.exp(params_hat[8] * xt_range),
    'Lag 48h': params_hat[9] + params_hat[10] * np.exp(params_hat[11] * xt_range),
    'Lag 168h': params_hat[12] + params_hat[13] * np.exp(params_hat[14] * xt_range)
}

plt.figure(figsize=(15, 10))
colors = ['blue', 'red', 'green', 'orange', 'purple']

for i, (name, effect) in enumerate(effects.items()):
    plt.subplot(2, 3, i+1)
    plt.plot(xt_range, effect, color=colors[i], linewidth=2)
    plt.xlabel('xt (normalisé)')
    plt.ylabel(f'Coefficient {name}')
    plt.title(f'Effet non-linéaire: {name}')
    
    if not np.isnan(tau_hats[i]):
        plt.axvline(tau_hats[i], color='red', linestyle='--', alpha=0.7, 
                   label=f'τ = {tau_hats[i]:.2f}')
        plt.legend()
    
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('effets_non_lineaires.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# QUALITÉ DU MODÈLE
# ============================================================
print("\n" + "="*120)
print("QUALITÉ DU MODÈLE")
print("="*120)

y_pred_orig = retransform_price(y_pred_train, a_train, b_train)
y_obs_orig = df_train['Prix'].values
rmse_orig = np.sqrt(mean_squared_error(y_obs_orig, y_pred_orig))
mae_orig = mean_absolute_error(y_obs_orig, y_pred_orig)
r2_orig = r2_score(y_obs_orig, y_pred_orig)

print(f"📊 PERFORMANCE DU MODÈLE SUR TRAIN:")
print(f"  • RMSE (original)    : {rmse_orig:.4f} €/MWh")
print(f"  • MAE (original)     : {mae_orig:.4f} €/MWh")
print(f"  • R² (original)      : {r2_orig:.4f}")
print(f"  • Bootstrap réussi   : {bootstrap_success}/{B} répliques")

# ============================================================
# PRÉDICTION MULTI-STEP
# ============================================================
print("\n" + "="*80)
print("PRÉDICTION MULTI-STEP")
print("="*80)

pred_list = []
dates_test = sorted(pd.to_datetime(df_test.index.date).unique())

for date_j in tqdm(dates_test, desc="Prédiction quotidienne"):
    day_start = pd.Timestamp(date_j)
    day_range = pd.date_range(day_start, day_start + pd.Timedelta(hours=23), freq='H')
    
    for ts in day_range:
        if ts not in df.index:
            continue
        
        try:
            ylag_1 = df.at[ts, 'Prix_1h_trans']
            ylag_24 = df.at[ts, 'Prix_24h_trans']
            ylag_48 = df.at[ts, 'Prix_48h_trans']
            ylag_168 = df.at[ts, 'Prix_168h_trans']
            xt_val = df.at[ts, 'xt']
        except KeyError:
            continue
        
        if np.any(pd.isna([ylag_1, ylag_24, ylag_48, ylag_168, xt_val])):
            continue
        
        ylags_arr = np.array([[ylag_1, ylag_24, ylag_48, ylag_168]])
        y_pred_trans = model_optimal_minimal(params_hat, np.array([xt_val]), ylags_arr)[0]
        y_pred_orig = retransform_price(y_pred_trans, a_train, b_train)
        y_obs_orig = df.at[ts, 'Prix']
        
        pred_list.append({
            'ts': ts,
            'y_pred_trans': y_pred_trans,
            'y_pred_orig': y_pred_orig,
            'y_obs_orig': y_obs_orig
        })

pred_df = pd.DataFrame(pred_list).set_index('ts').sort_index()
print(f"✓ Prédictions générées: {len(pred_df)} points")

# ============================================================
# CALIBRATION SUR VALIDATION
# ============================================================
print("\n" + "="*80)
print("CALIBRATION SUR VALIDATION")
print("="*80)

if isinstance(xt_val, np.ndarray):
    if xt_val.ndim == 1:
        xt_val = xt_val.reshape(-1, 1)
        print(f"✓ xt_val redimensionné: {xt_val.shape}")
else:
    xt_val = np.array(xt_val).reshape(-1, 1)
    print(f"✓ xt_val converti en array: {xt_val.shape}")

# Prédictions sur validation
y_pred_val = np.zeros(len(xt_val))
for i in tqdm(range(len(xt_val)), desc="Prédictions validation"):
    xt_i = xt_val[i:i+1]
    y_lags_i = y_lags_val[i:i+1]
    y_pred_val[i] = model_optimal_minimal(params_hat, xt_i, y_lags_i)[0]

residuals_val = y_val - y_pred_val
sigma_val = np.std(residuals_val)

print(f"σ résidus validation = {sigma_val:.4f}")

# Calibration du facteur d'incertitude
facteurs = np.arange(0.8, 2.0, 0.1)
coverages = []

for facteur in tqdm(facteurs, desc="Calibration facteur"):
    sigma_bruit = sigma_val * facteur
    couv_temp = []
    for i in range(min(200, len(y_pred_val))):
        sims = y_pred_val[i] + np.random.normal(0, sigma_bruit, 999)
        ci_low = np.percentile(sims, 2.5)
        ci_high = np.percentile(sims, 97.5)
        couv_temp.append((y_val[i] >= ci_low) and (y_val[i] <= ci_high))
    coverages.append(np.mean(couv_temp))

idx_opt = np.argmin(np.abs(np.array(coverages) - 0.95))
facteur_opt = facteurs[idx_opt]
SIGMA_EPSILON_CALIBRE = sigma_val * facteur_opt

print(f"✓ Facteur optimal: {facteur_opt:.2f}")
print(f"✓ σ calibré: {SIGMA_EPSILON_CALIBRE:.4f}")
print(f"✓ Couverture obtenue: {coverages[idx_opt]:.3%}")

# ============================================================
# INCERTITUDE DU MODÈLE PAR BOOTSTRAP
# ============================================================
print("\n" + "="*80)
print("INCERTITUDE DU MODÈLE PAR BOOTSTRAP")
print("="*80)

pred_df['model_uncertainty_std'] = np.nan

if len(bootstrap_params) >= 50:
    n_bootstrap_samples = min(500, len(bootstrap_params))

    for idx, row in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Incertitude modèle"):
        try:
            if pd.isna(row['y_pred_trans']):
                continue

            ts = idx
            if ts not in df.index:
                continue

            ylags_arr = np.array([[df.at[ts, 'Prix_1h_trans'],
                                   df.at[ts, 'Prix_24h_trans'],
                                   df.at[ts, 'Prix_48h_trans'],
                                   df.at[ts, 'Prix_168h_trans']]])
            xt_val = df.at[ts, 'xt']

            bootstrap_indices = np.random.choice(len(bootstrap_params), n_bootstrap_samples, replace=True)
            predictions_bootstrap = []

            for idx_b in bootstrap_indices:
                params_b = bootstrap_params[idx_b]
                y_pred_b = model_optimal_minimal(params_b[:15], np.array([xt_val]), ylags_arr)[0]
                predictions_bootstrap.append(y_pred_b)

            model_std = np.std(predictions_bootstrap, ddof=1)
            pred_df.at[idx, 'model_uncertainty_std'] = model_std

        except Exception as e:
            print(f"⚠️ Erreur à l'index {idx}: {e}")
            continue

    median_model_std = np.nanmedian(pred_df['model_uncertainty_std'])
    pred_df['model_uncertainty_std'].fillna(median_model_std, inplace=True)

    print(f"✓ Incertitude modèle - moyenne: {np.nanmean(pred_df['model_uncertainty_std']):.4f}")
else:
    sigma_approx = np.std(residuals_train_centered) * 0.5
    pred_df['model_uncertainty_std'] = sigma_approx
    print(f"⚠️ Bootstrap insuffisant, approximation: {sigma_approx:.4f}")

# ============================================================
# INFÉRENCE BOOTSTRAP POUR PRÉDICTIONS
# ============================================================
print("\n" + "="*80)
print("INFÉRENCE BOOTSTRAP - 9999 SIMULATIONS")
print("="*80)

B_sim = 9999
T_train = len(residuals_train_centered)

pred_df['bootstrap_mean_trans'] = np.nan
pred_df['bootstrap_std_trans'] = np.nan
pred_df['bootstrap_mean_orig'] = np.nan
pred_df['bootstrap_std_orig'] = np.nan
pred_df['bias_correction'] = np.nan
pred_df['prob_negative'] = np.nan
pred_df['ci_low_orig'] = np.nan
pred_df['ci_high_orig'] = np.nan

for idx, row in tqdm(pred_df.iterrows(), total=len(pred_df), desc="Simulations bootstrap"):
    try:
        p_hat_trans = row['y_pred_trans']
        model_std = row['model_uncertainty_std']

        if pd.isna(p_hat_trans) or pd.isna(model_std):
            continue

        ts = idx
        ylags_arr = np.array([[df.at[ts, 'Prix_1h_trans'],
                               df.at[ts, 'Prix_24h_trans'],
                               df.at[ts, 'Prix_48h_trans'],
                               df.at[ts, 'Prix_168h_trans']]])
        xt_val = df.at[ts, 'xt']
        
        p_star_trans = np.zeros(B_sim)
        
        for b in range(B_sim):
            eps_modele = np.random.normal(0, model_std)
            eps_bruit = np.random.normal(0, SIGMA_EPSILON_CALIBRE)
            p_star_trans[b] = p_hat_trans + eps_modele + eps_bruit

        bootstrap_mean_trans = np.mean(p_star_trans)
        bias_correction = bootstrap_mean_trans - p_hat_trans

        p_star_orig = retransform_price(p_star_trans, a_train, b_train)
        bootstrap_mean_orig = np.mean(p_star_orig)
        bootstrap_std_orig = np.std(p_star_orig, ddof=1)
        prob_negative = np.mean(p_star_orig < 0)

        ci_low_orig = np.percentile(p_star_orig, 2.5)
        ci_high_orig = np.percentile(p_star_orig, 97.5)

        pred_df.at[idx, 'bootstrap_mean_trans'] = bootstrap_mean_trans
        pred_df.at[idx, 'bootstrap_std_trans'] = np.std(p_star_trans, ddof=1)
        pred_df.at[idx, 'bootstrap_mean_orig'] = bootstrap_mean_orig
        pred_df.at[idx, 'bootstrap_std_orig'] = bootstrap_std_orig
        pred_df.at[idx, 'bias_correction'] = bias_correction
        pred_df.at[idx, 'prob_negative'] = prob_negative
        pred_df.at[idx, 'ci_low_orig'] = ci_low_orig
        pred_df.at[idx, 'ci_high_orig'] = ci_high_orig

    except Exception as e:
        print(f"⚠️ Erreur à l'index {idx}: {e}")
        continue

print(f"✓ Simulations terminées: {pred_df['bootstrap_mean_trans'].notna().sum()}/{len(pred_df)} points")

# ============================================================
# CORRECTION DU BIAIS
# ============================================================
print("\n" + "="*80)
print("📐 CORRECTION DU BIAIS")
print("="*80)

bias_mean_abs = np.mean(np.abs(pred_df['bias_correction'].dropna()))
bias_mean = pred_df['bias_correction'].mean()

print(f"\n📊 STATISTIQUES DU BIAIS (espace transformé):")
print(f"  • Biais moyen: {bias_mean:.6f}")
print(f"  • Biais absolu moyen: {bias_mean_abs:.6f}")

test_mask = pred_df.index.isin(df_test.index)

if bias_mean_abs > 0.01:
    print("⚠️ Biais non-négligeable - Correction appliquée")
    pred_df['y_pred_final_orig'] = pred_df['bootstrap_mean_orig']
    
    rmse_avant = np.sqrt(mean_squared_error(
        pred_df.loc[test_mask, 'y_obs_orig'], 
        pred_df.loc[test_mask, 'y_pred_orig']
    ))
    rmse_apres = np.sqrt(mean_squared_error(
        pred_df.loc[test_mask, 'y_obs_orig'], 
        pred_df.loc[test_mask, 'bootstrap_mean_orig']
    ))
    print(f"  • RMSE avant correction: {rmse_avant:.2f} €/MWh")
    print(f"  • RMSE après correction: {rmse_apres:.2f} €/MWh")
    print(f"  • Amélioration: {(rmse_avant - rmse_apres):.2f} €/MWh")
else:
    print("✓ Biais négligeable - Prédiction initiale conservée")
    pred_df['y_pred_final_orig'] = pred_df['y_pred_orig']

print(f"\n✓ Prédictions finales: {pred_df['y_pred_final_orig'].notna().sum()} points")

# ============================================================
# ÉVALUATION DES PERFORMANCES SUR TEST
# ============================================================
print("\n" + "="*80)
print("ÉVALUATION DES PERFORMANCES - TEST")
print("="*80)

test_eval = pred_df[test_mask].dropna(subset=['y_obs_orig', 'y_pred_final_orig', 'prob_negative'])

if len(test_eval) > 0:
    y_true = test_eval['y_obs_orig'].values
    y_pred = test_eval['y_pred_final_orig'].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) /
                         (np.abs(y_pred) + np.abs(y_true) + 1e-8))

    print(f"\n📊 MÉTRIQUES DE PERFORMANCE GLOBALES:")
    print(f"  • RMSE:  {rmse:.2f} €/MWh")
    print(f"  • MAE:   {mae:.2f} €/MWh")
    print(f"  • R²:    {r2:.4f}")
    print(f"  • SMAPE: {smape:.2f}%")

    mask_negatifs = y_true < 0
    if mask_negatifs.sum() > 0:
        rmse_neg = np.sqrt(mean_squared_error(y_true[mask_negatifs], y_pred[mask_negatifs]))
        mae_neg = mean_absolute_error(y_true[mask_negatifs], y_pred[mask_negatifs])
        bias_neg = np.mean(y_pred[mask_negatifs] - y_true[mask_negatifs])
        print(f"\n📉 PERFORMANCE SUR PRIX NÉGATIFS ({mask_negatifs.sum()} observations):")
        print(f"  • RMSE: {rmse_neg:.2f} €/MWh")
        print(f"  • MAE:  {mae_neg:.2f} €/MWh")
        print(f"  • Biais: {bias_neg:+.2f} €/MWh")

    coverage = ((y_true >= test_eval['ci_low_orig'].values) &
                (y_true <= test_eval['ci_high_orig'].values)).mean()
    width = (test_eval['ci_high_orig'] - test_eval['ci_low_orig']).mean()

    print(f"\n📊 INTERVALLES DE PRÉDICTION (95%):")
    print(f"  • Couverture: {coverage:.3%}")
    print(f"  • Largeur moyenne: {width:.2f} €/MWh")

    obs_neg_rate = mask_negatifs.mean()
    pred_neg_rate = test_eval['prob_negative'].mean()

    print(f"\n📉 PROBABILITÉS DE PRIX NÉGATIFS:")
    print(f"  • Taux observé: {obs_neg_rate:.3%}")
    print(f"  • Taux prédit:  {pred_neg_rate:.3%}")
    print(f"  • Écart:        {pred_neg_rate - obs_neg_rate:+.3%}")

# =============================================================
# VISUALISATIONS EXPLORATOIRES (SANS SEUILS)
# =============================================================
print("\n" + "="*80)
print("📈 VISUALISATIONS EXPLORATOIRES")
print("="*80)

# Préparation des colonnes
df_pred = pred_df.copy().sort_index()

if 'y_obs_orig' in df_pred.columns:
    df_pred['Prix_reel'] = df_pred['y_obs_orig']
else:
    print("⚠️ Colonne 'y_obs_orig' manquante")
    df_pred['Prix_reel'] = np.nan

if 'y_pred_final_orig' in df_pred.columns and df_pred['y_pred_final_orig'].notna().any():
    df_pred['Prix_pred_corrige'] = df_pred['y_pred_final_orig']
elif 'y_pred_orig' in df_pred.columns and df_pred['y_pred_orig'].notna().any():
    df_pred['Prix_pred_corrige'] = df_pred['y_pred_orig']
else:
    print("⚠️ Aucune colonne de prix prédit disponible")
    df_pred['Prix_pred_corrige'] = np.nan

if 'ci_low_final' in df_pred.columns and 'ci_high_final' in df_pred.columns:
    L_col, U_col = 'ci_low_final', 'ci_high_final'
elif 'ci_low_orig' in df_pred.columns and 'ci_high_orig' in df_pred.columns:
    L_col, U_col = 'ci_low_orig', 'ci_high_orig'
else:
    L_col = U_col = None

if 'prob_calibree' in df_pred.columns:
    df_pred['prob_negative'] = df_pred['prob_calibree']
    print("✓ Utilisation des probabilités calibrées")
elif 'prob_negative' not in df_pred.columns or not df_pred['prob_negative'].notna().any():
    df_pred['prob_negative'] = np.nan

# Graphiques
mask_obs = df_pred['Prix_reel'].notna() & df_pred['Prix_pred_corrige'].notna()
if not mask_obs.any():
    print("⚠️ Aucune observation valide pour les visualisations")
else:
    df_obs = df_pred.loc[mask_obs]

    # Graphique 1: Vue d'ensemble
    print("\n📈 Graphique 1: Vue d'ensemble")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    ax1.plot(df_obs.index, df_obs['Prix_reel'], 'k-', linewidth=1, label='Prix réel', alpha=0.8)
    ax1.plot(df_obs.index, df_obs['Prix_pred_corrige'], 'b-', linewidth=1, label='Prix prédit', alpha=0.8)
    if L_col and U_col:
        ax1.fill_between(df_obs.index, df_obs[L_col], df_obs[U_col],
                         color='blue', alpha=0.2, label='IC 95%')
    ax1.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Seuil 0 €/MWh')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylabel('Prix (€/MWh)')
    ax1.set_title('Observé vs Prédit avec IC 95%')
    ax1.legend(loc='upper left')

    if df_obs['prob_negative'].notna().any():
        ax2.bar(df_obs.index, df_obs['prob_negative'],
                color=plt.cm.RdYlGn_r(df_obs['prob_negative']), alpha=0.7, width=0.02)
        ax2.grid(True, alpha=0.3)
        ax2.set_ylabel('Probabilité de prix négatif')
        ax2.set_title('Probabilités Prédites')
        ax2.set_ylim(0, 1)
    else:
        ax2.text(0.5, 0.5, "Probabilités non disponibles", transform=ax2.transAxes,
                 ha='center', va='center', fontsize=12)
        ax2.axis('off')

    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('visualisation_1_vue_ensemble.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Graphique 2: Zoom sur une semaine
    print("\n📈 Graphique 2: Zoom sur une semaine")
    first_week_start = df_obs.index.min().ceil('D')
    week_end = first_week_start + pd.Timedelta(days=6, hours=23)
    df_week = df_pred.loc[first_week_start:week_end].copy()

    if len(df_week) > 0:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

        ax1.plot(df_week.index, df_week['Prix_reel'], 'ko-', linewidth=2,
                 markersize=4, label='Prix réel', alpha=0.8)
        ax1.plot(df_week.index, df_week['Prix_pred_corrige'], 'bs-', linewidth=2,
                 markersize=4, label='Prix prédit', alpha=0.8)
        if L_col and U_col:
            ax1.fill_between(df_week.index, df_week[L_col], df_week[U_col],
                             color='blue', alpha=0.2, label='IC 95%')
        ax1.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
        ax1.grid(True, alpha=0.3)
        ax1.set_ylabel('Prix (€/MWh)')
        ax1.set_title(f'Zoom sur une Semaine: {first_week_start.date()} à {week_end.date()}')
        ax1.legend(loc='upper left')

        if df_week['prob_negative'].notna().any():
            ax2.bar(df_week.index, df_week['prob_negative'],
                    color=plt.cm.RdYlGn_r(df_week['prob_negative']), alpha=0.7, width=0.02)
            ax2.grid(True, alpha=0.3)
            ax2.set_ylabel('Probabilité de prix négatif')
            ax2.set_xlabel('Date et Heure')
            ax2.set_title('Probabilités Prédites - Détail Horaire')
            ax2.set_ylim(0, 1)
        else:
            ax2.text(0.5, 0.5, "Probabilités non disponibles", transform=ax2.transAxes,
                     ha='center', va='center', fontsize=12)
            ax2.axis('off')

        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('visualisation_2_zoom_semaine.png', dpi=300, bbox_inches='tight')
        plt.show()

    # Graphique 3: Période de prix négatifs
    print("\n📈 Graphique 3: Focus sur période de prix négatifs")
    prix_neg_mask = df_pred['Prix_reel'] < 0
    if prix_neg_mask.any():
        df_temp = df_pred.copy()
        df_temp['groupe'] = (prix_neg_mask != prix_neg_mask.shift()).cumsum()
        groupes_neg = df_temp[prix_neg_mask].groupby('groupe').size()
        
        if len(groupes_neg) > 0:
            longest_group = groupes_neg.idxmax()
            periode_idx = df_temp[df_temp['groupe'] == longest_group].index
            debut = periode_idx.min() - pd.Timedelta(hours=24)
            fin = periode_idx.max() + pd.Timedelta(hours=24)
            df_focus = df_pred.loc[debut:fin].copy()

            if len(df_focus) > 0:
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

                ax1.plot(df_focus.index, df_focus['Prix_reel'], 'ko-',
                         linewidth=2, markersize=4, label='Prix réel', alpha=0.8)
                ax1.plot(df_focus.index, df_focus['Prix_pred_corrige'], 'bs-',
                         linewidth=2, markersize=4, label='Prix prédit', alpha=0.8)
                if L_col and U_col:
                    ax1.fill_between(df_focus.index, df_focus[L_col], df_focus[U_col],
                                     color='blue', alpha=0.2, label='IC 95%')
                ax1.axvspan(periode_idx.min(), periode_idx.max(), 
                           alpha=0.15, color='red', label='Période de prix négatifs')
                ax1.axhline(0, color='red', linestyle='--', linewidth=2, alpha=0.5)
                ax1.grid(True, alpha=0.3)
                ax1.set_ylabel('Prix (€/MWh)')
                ax1.set_title(f'Focus sur Période de Prix Négatifs ({len(periode_idx)} heures)')
                ax1.legend(loc='upper left')

                if df_focus['prob_negative'].notna().any():
                    ax2.bar(df_focus.index, df_focus['prob_negative'],
                            color=plt.cm.RdYlGn_r(df_focus['prob_negative']), 
                            alpha=0.7, width=0.02)
                    ax2.axvspan(periode_idx.min(), periode_idx.max(), 
                               alpha=0.15, color='red')
                    ax2.grid(True, alpha=0.3)
                    ax2.set_ylabel('Probabilité de prix négatif')
                    ax2.set_xlabel('Date et Heure')
                    ax2.set_title('Probabilités Prédites - Période Critique')
                    ax2.set_ylim(0, 1)
                else:
                    ax2.text(0.5, 0.5, "Probabilités non disponibles", transform=ax2.transAxes,
                             ha='center', va='center', fontsize=12)
                    ax2.axis('off')

                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.savefig('visualisation_3_periode_negative.png', dpi=300, bbox_inches='tight')
                plt.show()

                print(f"\n📊 STATISTIQUES DE LA PÉRIODE:")
                print(f"  • Durée: {len(periode_idx)} heures")
                print(f"  • Prix minimum: {df_focus.loc[periode_idx, 'Prix_reel'].min():.2f} €/MWh")
                print(f"  • Prix moyen: {df_focus.loc[periode_idx, 'Prix_reel'].mean():.2f} €/MWh")
                if df_focus['prob_negative'].notna().any():
                    print(f"  • Probabilité moyenne: {df_focus.loc[periode_idx, 'prob_negative'].mean():.3f}")
                    print(f"  • Probabilité maximale: {df_focus.loc[periode_idx, 'prob_negative'].max():.3f}")
    else:
        print("ℹ️ Aucune période de prix négatifs détectée")

print("\n" + "="*80)
print("✅ PARTIE 1 TERMINÉE - MODÈLE OPTIMISÉ ET VALIDÉ")
print("="*80)



# ============================================================
# BENCHMARKING - COMPARAISON AVEC AUTRES MODÈLES
# ============================================================
print("\n" + "="*80)
print("BENCHMARKING - COMPARAISON DE MODÈLES")
print("="*80)

from sklearn.ensemble import RandomForestRegressor
from hmmlearn import hmm
from sklearn.metrics import confusion_matrix

benchmark_results = {}

# Préparation des données
y_obs_test_orig = df_test['Prix'].values
y_train_trans = df_train['Prix_trans'].values
y_test_trans = df_test['Prix_trans'].values

# Benchmark 1: Modèle fondamental
X_train_fund = sm.add_constant(df_train[['xt']])
X_test_fund = sm.add_constant(df_test[['xt']])
fund_model = sm.OLS(y_train_trans, X_train_fund, missing='drop').fit()
y_pred_test_fund = retransform_price(fund_model.predict(X_test_fund), a_train, b_train)
rmse_test_fund = np.sqrt(mean_squared_error(y_obs_test_orig, y_pred_test_fund))

benchmark_results['Fondamental'] = {
    'test': {'rmse': rmse_test_fund},
    'predictions_test': y_pred_test_fund
}

# Benchmark 2: ARX Linéaire (avec 3 lags)
X_train_arx = sm.add_constant(df_train[['xt', 'Prix_1h_trans', 'Prix_24h_trans', 'Prix_168h_trans']])
X_test_arx = sm.add_constant(df_test[['xt', 'Prix_1h_trans', 'Prix_24h_trans', 'Prix_168h_trans']])
arx_model = sm.OLS(y_train_trans, X_train_arx, missing='drop').fit()
y_pred_test_arx = retransform_price(arx_model.predict(X_test_arx), a_train, b_train)
rmse_test_arx = np.sqrt(mean_squared_error(y_obs_test_orig, y_pred_test_arx))

benchmark_results['ARX Linéaire'] = {
    'test': {'rmse': rmse_test_arx},
    'predictions_test': y_pred_test_arx
}

# Benchmark 3: Random Forest
X_train_rf = df_train[['xt', 'Prix_1h_trans', 'Prix_24h_trans', 'Prix_168h_trans']]
X_test_rf = df_test[['xt', 'Prix_1h_trans', 'Prix_24h_trans', 'Prix_168h_trans']]
rf_model = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train_trans)
y_pred_test_rf = retransform_price(rf_model.predict(X_test_rf), a_train, b_train)
rmse_test_rf = np.sqrt(mean_squared_error(y_obs_test_orig, y_pred_test_rf))

benchmark_results['Random Forest'] = {
    'test': {'rmse': rmse_test_rf},
    'predictions_test': y_pred_test_rf
}

# Benchmark 4: HMM
try:
    residuals_hmm = y_train_trans - arx_model.predict(X_train_arx)
    if hasattr(residuals_hmm, 'values'):
        residuals_hmm = residuals_hmm.values
    residuals_hmm = residuals_hmm[~np.isnan(residuals_hmm)].reshape(-1, 1)

    hmm_model = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=1000, random_state=42)
    hmm_model.fit(residuals_hmm)

    y_pred_arx_test = arx_model.predict(X_test_arx)
    last_state = hmm_model.predict(residuals_hmm[-1].reshape(1, -1))[0]
    n_test = len(X_test_arx)
    states_test = np.zeros(n_test, dtype=int)
    states_test[0] = last_state

    for t in range(1, n_test):
        states_test[t] = np.random.choice([0, 1], p=hmm_model.transmat_[states_test[t-1]])

    y_pred_test_hmm = retransform_price(y_pred_arx_test + hmm_model.means_[states_test, 0], a_train, b_train)
    rmse_test_hmm = np.sqrt(mean_squared_error(y_obs_test_orig, y_pred_test_hmm))

    benchmark_results['HMM (2 régimes)'] = {
        'test': {'rmse': rmse_test_hmm},
        'predictions_test': y_pred_test_hmm
    }
except Exception as e:
    print(f"⚠️ HMM non disponible: {e}")

# Benchmark 5: Votre modèle
rmse_test_nl = rmse  # Calculé plus haut
benchmark_results['Modèle Optimal (13p)'] = {
    'test': {'rmse': rmse_test_nl},
    'predictions_test': y_pred,
    'prob_negative': test_eval['prob_negative'].values if 'prob_negative' in test_eval.columns else None
}

# Classement
comparison_data = []
for model_name, results in benchmark_results.items():
    if 'test' in results and 'rmse' in results['test']:
        comparison_data.append({
            'Modèle': model_name,
            'RMSE_test': results['test']['rmse']
        })

df_comparison = pd.DataFrame(comparison_data).sort_values('RMSE_test')

print("\n" + "-"*80)
print("🏆 CLASSEMENT DES MODÈLES")
print("-"*80)
print(df_comparison.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

# =============================================================
# VISUALISATIONS COMPARATIVES
# =============================================================
print("\n" + "="*80)
print("📈 VISUALISATIONS COMPARATIVES")
print("="*80)

# 1. Graphique série temporelle - comparaison sur une semaine
print("\n→ Génération du graphique comparatif...")

# Sélectionner une semaine représentative (première semaine de test)
test_start = df_test.index.min()
week_end = test_start + pd.Timedelta(days=6, hours=23)
week_mask = (df_test.index >= test_start) & (df_test.index <= week_end)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Graphique 1: Comparaison des prédictions
ax1.plot(df_test.index[week_mask], y_obs_test_orig[week_mask],
         'k-', linewidth=2, label='Prix Réel', alpha=0.9, zorder=10)

# Modèles à afficher
models_to_plot = ['ARX Linéaire', 'Random Forest', 'HMM (2 régimes)', 'Modèle Optimal (13p)']
colors_plot = ['blue', 'green', 'orange', 'purple']
styles = ['-', '--', '-.', ':']

for model_name, color, style in zip(models_to_plot, colors_plot, styles):
    if model_name in benchmark_results:
        pred = benchmark_results[model_name]['predictions_test']
        if len(pred) > 0:
            if len(pred) == len(df_test):
                pred_week = pred[week_mask]
            else:
                pred_series = pd.Series(pred, index=df_test.index[:len(pred)])
                pred_week = pred_series[week_mask].values if len(pred_series) > 0 else []
            
            if len(pred_week) > 0:
                linewidth = 2.5 if model_name == 'Modèle Optimal (13p)' else 1.5
                alpha = 0.9 if model_name == 'Modèle Optimal (13p)' else 0.7
                ax1.plot(df_test.index[week_mask][:len(pred_week)], pred_week,
                        color=color, linestyle=style, linewidth=linewidth,
                        label=f"{model_name} (RMSE={benchmark_results[model_name]['test']['rmse']:.1f})",
                        alpha=alpha)

ax1.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Date et Heure', fontsize=11)
ax1.set_ylabel('Prix (€/MWh)', fontsize=11)
ax1.set_title('Comparaison des Modèles - Première Semaine de Test (2024)',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='best')

# Graphique 2: Zoom sur les écarts
ax2.plot(df_test.index[week_mask], y_obs_test_orig[week_mask] - benchmark_results['Modèle Optimal (13p)']['predictions_test'][week_mask],
         'r-', linewidth=1, label='Erreur Modèle Optimal', alpha=0.7)
ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax2.axhline(np.std(y_obs_test_orig[week_mask] - benchmark_results['Modèle Optimal (13p)']['predictions_test'][week_mask]), 
            color='red', linestyle='--', alpha=0.5, label='±1 écart-type')
ax2.axhline(-np.std(y_obs_test_orig[week_mask] - benchmark_results['Modèle Optimal (13p)']['predictions_test'][week_mask]), 
            color='red', linestyle='--', alpha=0.5)
ax2.set_xlabel('Date et Heure', fontsize=11)
ax2.set_ylabel('Erreur de prédiction (€/MWh)', fontsize=11)
ax2.set_title('Erreurs du Modèle Optimal - Première Semaine de Test')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=300, bbox_inches='tight')
plt.show()


# ============================================================
# APPLICATION OPÉRATIONNELLE - VALEUR ÉCONOMIQUE
# ============================================================
print("\n" + "="*80)
print("💰 APPLICATION OPÉRATIONNELLE - VALEUR POUR L'ENTREPRISE")
print("="*80)

import os

path_ppa_data = 'productionCGN_20_25.csv' # chemin relatif pour le dépôt

# Liste des parcs PPA/Marché (anonymisée pour le dépôt public)
parcs_PPA_Marche = [
    'PARC A', 
    'PARC B', 
    'PARC C', 
    'PARC D', 
    'PARC E', 
    'PARC F', 
    'PARC G', 
    'PARC H', 
    'PARC I', 
    'PARC J', 
    'PARC K', 
]

if os.path.exists(path_ppa_data):
    print("✓ Données PPA détectées — Exécution de la valorisation économique réelle.")
    df_parcs = pd.read_csv(path_ppa_data)
    df_parcs['Date and time'] = pd.to_datetime(df_parcs['Date and time'])
    df_parcs = df_parcs.set_index('Date and time')
    df_parcs = df_parcs.loc['2024-01-01':'2024-12-31']
    
    parcs_disponibles = [p for p in parcs_PPA_Marche if p in df_parcs.columns]
    production_totale = df_parcs[parcs_disponibles].sum(axis=1) / 1000

else:
    print("⚠️ Données PPA confidentielles non fournies dans le dépôt de réplication.")
    print("   Génération d'une chronique de production synthétique à titre d'illustration...")
    
    # Création d'une série de production synthétique pour que le code public s'exécute de bout en bout
    np.random.seed(42)
    common_index_synth = pd.date_range("2024-01-01 00:00:00", "2024-12-31 23:00:00", freq='H')
    simulated_prod = np.random.weibull(a=2, size=len(common_index_synth)) * 50 # MW
    
    production_totale = pd.Series(simulated_prod, index=common_index_synth)
    parcs_disponibles = ['PARC A (Synthétique)']
    df_parcs = pd.DataFrame({'PARC A (Synthétique)': simulated_prod * 1000}, index=common_index_synth)

print(f"✓ {len(parcs_disponibles)} parc(s) pris en compte pour l'analyse.")

# ============================================================
# ALIGNEMENT AVEC LES PRÉDICTIONS
# ============================================================
print("\n→ Alignement avec les prédictions...")

pred_2024 = pred_df.loc['2024-01-01':'2024-12-31'].copy()
common_index = pred_2024.index.intersection(production_totale.index)
arbitrage_df = pd.DataFrame(index=common_index)

# Variables de base
arbitrage_df['Prix_Reel'] = df.loc[common_index, 'Prix']
arbitrage_df['Prod_MW'] = production_totale.loc[common_index]
arbitrage_df['Prod_GWh'] = arbitrage_df['Prod_MW'] / 1000
arbitrage_df['Revenu_Reel'] = arbitrage_df['Prod_MW'] * arbitrage_df['Prix_Reel']

# Probabilités du modèle
if 'prob_finale' in pred_df.columns:
    arbitrage_df['Prob_Negative'] = pred_2024.loc[common_index, 'prob_finale']
    print("✓ Utilisation des probabilités Platt calibrées")
elif 'prob_calibree' in pred_df.columns:
    arbitrage_df['Prob_Negative'] = pred_2024.loc[common_index, 'prob_calibree']
    print("✓ Utilisation des probabilités calibrées")
else:
    arbitrage_df['Prob_Negative'] = pred_2024.loc[common_index, 'prob_negative']
    print("✓ Utilisation des probabilités brutes")

# Ajouter les prédictions des benchmarks
for model_name in benchmark_results:
    if 'predictions_test' in benchmark_results[model_name]:
        preds = benchmark_results[model_name]['predictions_test']
        if len(preds) == len(df_test):
            preds_series = pd.Series(preds, index=df_test.index)
            preds_2024 = preds_series[preds_series.index.isin(common_index)]
            if len(preds_2024) == len(common_index):
                arbitrage_df[f'Prix_Predit_{model_name}'] = preds_2024

print(f"✓ {len(arbitrage_df)} heures analysées en 2024")
print(f"✓ Production totale 2024: {arbitrage_df['Prod_MW'].sum():,.0f} MWh")
print(f"✓ CA potentiel: {arbitrage_df['Revenu_Reel'].sum():,.0f} €")
    
# ============================================================
# ANALYSE DES PERTES SUR PRIX NÉGATIFS
# ============================================================
print("\n" + "="*80)
print("📉 ANALYSE DES PERTES SUR PRIX NÉGATIFS - SITUATION ACTUELLE")
print("="*80)

mask_negatifs = arbitrage_df['Prix_Reel'] < 0
nb_heures_neg = mask_negatifs.sum()
production_neg = arbitrage_df.loc[mask_negatifs, 'Prod_MW'].sum()
pertes_subies = arbitrage_df.loc[mask_negatifs, 'Revenu_Reel'].sum()
prix_neg_moyen = arbitrage_df.loc[mask_negatifs, 'Prix_Reel'].mean()

print(f"\n📊 STATISTIQUES GLOBALES 2024 ({len(parcs_disponibles)} parcs cumulés):")
print(f"  • Heures à prix négatifs: {nb_heures_neg} h ({nb_heures_neg/len(arbitrage_df)*100:.2f}% du temps)")
print(f"  • Production en période négative: {production_neg:,.0f} MWh ({production_neg/arbitrage_df['Prod_MW'].sum()*100:.2f}% de la production)")
print(f"  • Prix négatif moyen: {prix_neg_moyen:.2f} €/MWh")
print(f"  • PERTES TOTALES SUBIES: {pertes_subies:,.0f} € ({pertes_subies/1e6:.2f} M€)")

# ============================================================
# RECHERCHE DU SEUIL OPTIMAL (probabilités)
# ============================================================
print("\n" + "="*80)
print("🎯 RECHERCHE DU SEUIL OPTIMAL SUR PROBABILITÉS")
print("="*80)

seuils_test = np.concatenate([
    np.arange(0.01, 0.20, 0.005),
    np.arange(0.20, 0.40, 0.01),
    np.arange(0.40, 0.70, 0.02),
    np.arange(0.70, 0.91, 0.05)
])
seuils_test = np.unique(np.round(seuils_test, 3))

print(f"🔍 Grille: {len(seuils_test)} seuils [1% - 90%]")

resultats_seuils = pd.DataFrame(index=seuils_test, columns=[
    'Gain_€', 'Gain_relatif_%', 'Recuperation_%', 'Heures_arret', 'Prod_perdue_MWh',
    'Pertes_evitees_€', 'Cout_fausses_alertes_€', 
    'VP', 'FP', 'VN', 'FN', 'Precision_%', 'Rappel_%', 'F1_score'
])

for seuil in tqdm(seuils_test, desc="Test des seuils"):
    decision = arbitrage_df['Prob_Negative'] > seuil
    prod_opt = np.where(decision, 0, arbitrage_df['Prod_MW'])
    revenu_opt = prod_opt * arbitrage_df['Prix_Reel']
    
    VP = (decision & mask_negatifs).sum()
    FP = (decision & ~mask_negatifs).sum()
    VN = (~decision & ~mask_negatifs).sum()
    FN = (~decision & mask_negatifs).sum()
    
    pertes_evitees = -arbitrage_df.loc[decision & mask_negatifs, 'Revenu_Reel'].sum()
    cout_fausses = arbitrage_df.loc[decision & ~mask_negatifs, 'Revenu_Reel'].sum()
    gain_net = revenu_opt.sum() - arbitrage_df['Revenu_Reel'].sum()
    gain_relatif = gain_net / abs(pertes_subies) * 100 if pertes_subies != 0 else 0
    recuperation = (pertes_evitees / abs(pertes_subies)) * 100 if pertes_subies != 0 else 0
    
    precision = VP / (VP + FP) * 100 if (VP + FP) > 0 else 0
    rappel = VP / (VP + FN) * 100 if (VP + FN) > 0 else 0
    f1 = 2 * precision * rappel / (precision + rappel) / 100 if (precision + rappel) > 0 else 0
    
    resultats_seuils.loc[seuil] = {
        'Gain_€': gain_net,
        'Gain_relatif_%': gain_relatif,
        'Recuperation_%': recuperation,
        'Heures_arret': decision.sum(),
        'Prod_perdue_MWh': arbitrage_df.loc[decision, 'Prod_MW'].sum(),
        'Pertes_evitees_€': pertes_evitees,
        'Cout_fausses_alertes_€': cout_fausses,
        'VP': VP, 'FP': FP, 'VN': VN, 'FN': FN,
        'Precision_%': precision,
        'Rappel_%': rappel,
        'F1_score': f1
    }

idx_opt = resultats_seuils['Gain_€'].idxmax()
seuil_optimal = idx_opt
gain_optimal = resultats_seuils.loc[idx_opt, 'Gain_€']

print(f"\n✅ SEUIL OPTIMAL: {seuil_optimal:.1%}")
print(f"✅ GAIN MAXIMAL: {gain_optimal:,.0f} € ({gain_optimal/1e6:.2f} M€)")
print(f"✅ PERTES RÉCUPÉRÉES: {resultats_seuils.loc[idx_opt, 'Recuperation_%']:.1f}%")
print(f"✅ PRÉCISION/RAPPEL: {resultats_seuils.loc[idx_opt, 'Precision_%']:.1f}% / {resultats_seuils.loc[idx_opt, 'Rappel_%']:.1f}%")

# ============================================================
# OPTIMISATION DES SEUILS POUR LES MODÈLES BENCHMARK
# ============================================================
print("\n" + "="*80)
print("📊 OPTIMISATION DES SEUILS SUR MODÈLES BENCHMARK")
print("="*80)

def optimiser_seuil_benchmark(pred_series, prod, prix, mask_neg, pertes_subies, n_percentiles=100):
    """
    Optimise le seuil sur la prédiction d'un modèle benchmark.
    Retourne dict des métriques au seuil optimal (gain max).
    """
    pred_clean = pred_series.dropna()
    if len(pred_clean) == 0:
        return None
    
    seuils = np.percentile(pred_clean, np.linspace(1, 99, n_percentiles))
    seuils = np.unique(np.append(seuils, [pred_clean.min(), pred_clean.max()]))
    
    meilleur_gain = -np.inf
    meilleur_res = None
    
    for s in seuils:
        decision = pred_series < s
        if decision.sum() == 0:
            continue
        prod_opt = np.where(decision, 0, prod)
        gain = (prod_opt * prix).sum() - (prod * prix).sum()
        
        if gain > meilleur_gain:
            meilleur_gain = gain
            VP = (decision & mask_neg).sum()
            FP = (decision & ~mask_neg).sum()
            FN = (~decision & mask_neg).sum()
            pertes_evitees = -((prod * prix)[decision & mask_neg]).sum()
            cout_fausses = ((prod * prix)[decision & ~mask_neg]).sum()
            recuperation = (pertes_evitees / abs(pertes_subies)) * 100 if pertes_subies != 0 else 0
            precision = VP / (VP + FP) * 100 if (VP+FP)>0 else 0
            rappel = VP / (VP + FN) * 100 if (VP+FN)>0 else 0
            
            meilleur_res = {
                'seuil': s,
                'gain': gain,
                'VP': VP, 'FP': FP, 'FN': FN,
                'pertes_evitees': pertes_evitees,
                'cout_fausses': cout_fausses,
                'recuperation': recuperation,
                'precision': precision,
                'rappel': rappel,
                'heures_arret': decision.sum()
            }
    return meilleur_res

benchmark_models = [
    ('Random Forest', 'Prix_Predit_Random Forest'),
    ('ARX Linéaire', 'Prix_Predit_ARX Linéaire'),
    ('HMM', 'Prix_Predit_HMM (2 régimes)')
]

results_benchmark_opt = []

for nom, col in benchmark_models:
    if col in arbitrage_df.columns:
        opt = optimiser_seuil_benchmark(
            arbitrage_df[col], 
            arbitrage_df['Prod_MW'], 
            arbitrage_df['Prix_Reel'],
            mask_negatifs, 
            pertes_subies
        )
        if opt is not None:
            results_benchmark_opt.append({
                'Modèle': f'{nom} (seuil={opt["seuil"]:.2f} €/MWh)',
                'Gain_€': opt['gain'],
                'Gain_M€': opt['gain']/1e6,
                'Gain_relatif_%': opt['gain']/abs(pertes_subies)*100,
                'Recuperation_%': opt['recuperation'],
                'Heures_arret': opt['heures_arret'],
                'Pertes_evitees_€': opt['pertes_evitees'],
                'Fausses_alertes': opt['FP'],
                'Cout_fausses_€': opt['cout_fausses'],
                'VP': opt['VP'],
                'FP': opt['FP'],
                'FN': opt['FN'],
                'Precision_%': opt['precision'],
                'Rappel_%': opt['rappel']
            })
            print(f"✓ {nom} optimisé : seuil={opt['seuil']:.2f}, gain={opt['gain']:,.0f}€")
        else:
            print(f"⚠️ Échec optimisation {nom}")

# ============================================================
# TABLEAU COMPARATIF FINAL (AVEC TOUS MODÈLES OPTIMISÉS)
# ============================================================
print("\n" + "="*120)
print(f"🏆 CLASSEMENT ÉCONOMIQUE - TOUS MODÈLES OPTIMISÉS ({len(parcs_disponibles)} parcs cumulés)")
print("="*120)

resultats_economiques = []

# Modèle probabiliste (déjà optimisé)
decision_opt = arbitrage_df['Prob_Negative'] > seuil_optimal
prod_opt = np.where(decision_opt, 0, arbitrage_df['Prod_MW'])
gain_opt = (prod_opt * arbitrage_df['Prix_Reel']).sum() - arbitrage_df['Revenu_Reel'].sum()
VP_opt = (decision_opt & mask_negatifs).sum()
FP_opt = (decision_opt & ~mask_negatifs).sum()
FN_opt = (~decision_opt & mask_negatifs).sum()
pertes_evitees_opt = -arbitrage_df.loc[decision_opt & mask_negatifs, 'Revenu_Reel'].sum()
cout_fausses_opt = arbitrage_df.loc[decision_opt & ~mask_negatifs, 'Revenu_Reel'].sum()
recuperation_opt = (pertes_evitees_opt / abs(pertes_subies)) * 100 if pertes_subies != 0 else 0
precision_opt = VP_opt / (VP_opt + FP_opt) * 100 if (VP_opt+FP_opt)>0 else 0
rappel_opt = VP_opt / (VP_opt + FN_opt) * 100 if (VP_opt+FN_opt)>0 else 0

resultats_economiques.append({
    'Modèle': f'Probabiliste (seuil={seuil_optimal:.1%})',
    'Gain_€': gain_opt,
    'Gain_M€': gain_opt/1e6,
    'Gain_relatif_%': gain_opt/abs(pertes_subies)*100,
    'Recuperation_%': recuperation_opt,
    'Heures_arret': decision_opt.sum(),
    'Pertes_evitees_€': pertes_evitees_opt,
    'Fausses_alertes': FP_opt,
    'Cout_fausses_€': cout_fausses_opt,
    'VP': VP_opt,
    'FP': FP_opt,
    'FN': FN_opt,
    'Precision_%': precision_opt,
    'Rappel_%': rappel_opt
})

# Ajouter les benchmarks optimisés
resultats_economiques.extend(results_benchmark_opt)

# Modèle naïf (arrêt systématique)
decision_naif = pd.Series(True, index=arbitrage_df.index)
gain_naif = -arbitrage_df['Revenu_Reel'].sum()
pertes_evitees_naif = -pertes_subies
cout_fausses_naif = arbitrage_df.loc[~mask_negatifs, 'Revenu_Reel'].sum()
recuperation_naif = 100.0 if pertes_subies != 0 else 0
precision_naif = mask_negatifs.sum() / len(arbitrage_df) * 100

resultats_economiques.append({
    'Modèle': 'Arrêt systématique',
    'Gain_€': gain_naif,
    'Gain_M€': gain_naif/1e6,
    'Gain_relatif_%': gain_naif/abs(pertes_subies)*100,
    'Recuperation_%': recuperation_naif,
    'Heures_arret': len(arbitrage_df),
    'Pertes_evitees_€': pertes_evitees_naif,
    'Fausses_alertes': (~mask_negatifs).sum(),
    'Cout_fausses_€': cout_fausses_naif,
    'VP': mask_negatifs.sum(),
    'FP': (~mask_negatifs).sum(),
    'FN': 0,
    'Precision_%': precision_naif,
    'Rappel_%': 100.0
})

df_economique = pd.DataFrame(resultats_economiques).sort_values('Gain_€', ascending=False)

print(df_economique[[
    'Modèle', 'Gain_M€', 'Gain_relatif_%', 'Recuperation_%',
    'Heures_arret', 'Pertes_evitees_€', 'Fausses_alertes', 'Cout_fausses_€',
    'Precision_%', 'Rappel_%'
]].to_string(index=False, float_format=lambda x: f'{x:,.2f}' if 'M€' in str(x) or abs(x) > 1000 else f'{x:.1f}'))


# ============================================================
# INTERVALLE DE CONFIANCE SUR LE GAIN (BOOTSTRAP)
# ============================================================
print("\n" + "="*80)
print("📊 INTERVALLE DE CONFIANCE DU GAIN (meilleur modèle)")
print("="*80)

best_model = df_economique.iloc[0]
best_model_name = best_model['Modèle']
best_gain = best_model['Gain_€']

# Reconstruire la décision du meilleur modèle
if 'Probabiliste' in best_model_name:
    decision_best = arbitrage_df['Prob_Negative'] > seuil_optimal
elif 'Random Forest' in best_model_name:
    seuil = float(best_model_name.split('seuil=')[1].split()[0])
    decision_best = arbitrage_df['Prix_Predit_Random Forest'] < seuil
elif 'ARX' in best_model_name:
    seuil = float(best_model_name.split('seuil=')[1].split()[0])
    decision_best = arbitrage_df['Prix_Predit_ARX Linéaire'] < seuil
elif 'HMM' in best_model_name:
    seuil = float(best_model_name.split('seuil=')[1].split()[0])
    decision_best = arbitrage_df['Prix_Predit_HMM (2 régimes)'] < seuil
else:
    decision_best = None

if decision_best is not None:
    n_boot = 1000
    gains_boot = []
    indices = np.arange(len(decision_best))
    for _ in range(n_boot):
        idx = np.random.choice(indices, size=len(indices), replace=True)
        gain = ( (np.where(decision_best.iloc[idx], 0, arbitrage_df['Prod_MW'].iloc[idx]) * arbitrage_df['Prix_Reel'].iloc[idx] ).sum() 
                - arbitrage_df['Revenu_Reel'].iloc[idx].sum() )
        gains_boot.append(gain)
    ci_low, ci_high = np.percentile(gains_boot, [2.5, 97.5])
    print(f"Meilleur modèle : {best_model_name}")
    print(f"Gain estimé : {best_gain:,.0f} €")
    print(f"IC 95% : [{ci_low:,.0f} € , {ci_high:,.0f} €]")

# ============================================================
# ANALYSE PAR PARC INDIVIDUEL (AVEC PRODUCTION)
# ============================================================
print("\n" + "="*80)
print("🔍 ANALYSE DÉTAILLÉE PAR PARC INDIVIDUEL")
print("="*80)

resultats_par_parc = []

for parc in tqdm(parcs_disponibles, desc="Analyse individuelle"):
    
    prod_parc_mwh = df_parcs[parc] / 1000
    prod_parc = prod_parc_mwh.loc[common_index]
    revenu_parc = prod_parc * arbitrage_df['Prix_Reel']
    pertes_parc = revenu_parc[revenu_parc < 0].sum()
    
    if pertes_parc >= 0:
        continue
    
    # Modèle probabiliste (seuil global)
    decision = arbitrage_df['Prob_Negative'] > seuil_optimal
    prod_opt = np.where(decision, 0, prod_parc)
    gain = (prod_opt * arbitrage_df['Prix_Reel']).sum() - revenu_parc.sum()
    VP = (decision & mask_negatifs).sum()
    FP = (decision & ~mask_negatifs).sum()
    FN = (~decision & mask_negatifs).sum()
    pertes_evitees = -revenu_parc[decision & mask_negatifs].sum()
    taux_recup = (pertes_evitees / abs(pertes_parc)) * 100 if pertes_parc != 0 else 0
    
    resultats_par_parc.append({
        'Parc': parc,
        'Modèle': 'Probabiliste (seuil global)',
        'Gain_€': gain,
        'Recuperation_%': taux_recup,
        'Heures_arret': decision.sum(),
        'VP': VP,
        'FP': FP,
        'FN': FN,
        'Prod_MWh': prod_parc.sum(),
        'Prod_GWh': prod_parc.sum()/1000,
        'Pertes_parc_€': pertes_parc
    })
    
    # Random Forest (seuil optimal benchmark)
    if 'Prix_Predit_Random Forest' in arbitrage_df.columns:
        # Chercher le meilleur seuil pour ce parc
        opt = optimiser_seuil_benchmark(
            arbitrage_df['Prix_Predit_Random Forest'],
            prod_parc,
            arbitrage_df['Prix_Reel'],
            mask_negatifs,
            pertes_parc,
            n_percentiles=50  # moins fin pour vitesse
        )
        if opt is not None:
            resultats_par_parc.append({
                'Parc': parc,
                'Modèle': f'Random Forest (seuil={opt["seuil"]:.1f})',
                'Gain_€': opt['gain'],
                'Recuperation_%': opt['recuperation'],
                'Heures_arret': opt['heures_arret'],
                'VP': opt['VP'],
                'FP': opt['FP'],
                'FN': opt['FN'],
                'Prod_MWh': prod_parc.sum(),
                'Prod_GWh': prod_parc.sum()/1000,
                'Pertes_parc_€': pertes_parc
            })
    
    # Arrêt systématique
    gain_naif = -revenu_parc.sum()
    resultats_par_parc.append({
        'Parc': parc,
        'Modèle': 'Arrêt systématique',
        'Gain_€': gain_naif,
        'Recuperation_%': 100.0,
        'Heures_arret': len(arbitrage_df),
        'VP': mask_negatifs.sum(),
        'FP': (~mask_negatifs).sum(),
        'FN': 0,
        'Prod_MWh': prod_parc.sum(),
        'Prod_GWh': prod_parc.sum()/1000,
        'Pertes_parc_€': pertes_parc
    })

df_par_parc = pd.DataFrame(resultats_par_parc)

# ============================================================
# MEILLEUR MODÈLE PAR PARC
# ============================================================
print("\n" + "="*120)
print("📊 MEILLEUR MODÈLE POUR CHAQUE PARC")
print("="*120)

idx_meilleur = df_par_parc.groupby('Parc')['Gain_€'].idxmax()
meilleurs_par_parc = df_par_parc.loc[idx_meilleur].sort_values('Gain_€', ascending=False)
meilleurs_par_parc['Gain_€_par_MWh'] = meilleurs_par_parc['Gain_€'] / meilleurs_par_parc['Prod_MWh']

print(meilleurs_par_parc[[
    'Parc', 'Prod_GWh', 'Pertes_parc_€', 'Modèle', 'Gain_€', 
    'Recuperation_%', 'Heures_arret', 'VP', 'FP', 'FN', 'Gain_€_par_MWh'
]].to_string(index=False, float_format=lambda x: f'{x:,.2f}' if 'GWh' in str(x) else f'{x:,.0f}' if abs(x) > 1000 else f'{x:.1f}'))

# ============================================================
# BOXPLOT DES GAINS PAR PARC
# ============================================================
print("\n" + "="*80)
print("📈 BOXPLOT - DISTRIBUTION DES GAINS PAR PARC")
print("="*80)

parcs_list = df_par_parc['Parc'].unique()
data_boxplot = [df_par_parc[df_par_parc['Parc'] == p]['Gain_€'].values for p in parcs_list]

plt.figure(figsize=(14, 6))
bp = plt.boxplot(data_boxplot, labels=[p[:15] for p in parcs_list], 
                 patch_artist=True, showmeans=True)
for box in bp['boxes']:
    box.set_facecolor('lightblue')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Gain (€)')
plt.title('Distribution des gains par parc (tous modèles)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('boxplot_gains_par_parc.png', dpi=300)
plt.show()

# ============================================================
# STATISTIQUES DE VARIABILITÉ
# ============================================================
print("\n" + "="*80)
print("📊 STATISTIQUES DE VARIABILITÉ ENTRE PARCS")
print("="*80)

stats_gain = meilleurs_par_parc['Gain_€'].describe(percentiles=[.25, .5, .75])
print("\nDistribution des gains optimaux par parc :")
print(stats_gain.apply(lambda x: f'{x:,.0f} €'))

print("\n🏆 Modèles les plus souvent retenus :")
compteur = meilleurs_par_parc['Modèle'].value_counts()
for modele, count in compteur.items():
    print(f"  • {modele}: {count} parcs ({count/len(meilleurs_par_parc)*100:.1f}%)")

# ============================================================
# SYNTHÈSE FINALE
# ============================================================
print("\n" + "="*120)
print("📋 SYNTHÈSE FINALE - VALEUR ÉCONOMIQUE")
print("="*120)

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 RÉSULTATS CLÉS - PORTEFEUILLE {len(parcs_disponibles)} PARCS PPA

📉 SITUATION ACTUELLE :
   • Heures négatives : {nb_heures_neg} h ({nb_heures_neg/len(arbitrage_df)*100:.1f}%)
   • Production concernée : {production_neg:,.0f} MWh
   • PERTES SUBIES : {pertes_subies:,.0f} € ({pertes_subies/1e6:.2f} M€)

🏆 MEILLEURE STRATÉGIE GLOBALE :
   • {df_economique.iloc[0]['Modèle']}
   • Gain : {df_economique.iloc[0]['Gain_€']:,.0f} € ({df_economique.iloc[0]['Gain_M€']:.2f} M€)
   • Pertes évitées : {df_economique.iloc[0]['Pertes_evitees_€']:,.0f} €
   • Pertes récupérées : {df_economique.iloc[0]['Recuperation_%']:.1f}%
   • Précision / Rappel : {df_economique.iloc[0]['Precision_%']:.1f}% / {df_economique.iloc[0]['Rappel_%']:.1f}%

📈 ANALYSE PAR PARC :
   • Parcs analysés : {len(meilleurs_par_parc)} (ceux ayant des pertes)
   • Gain total si optimisation individuelle : {meilleurs_par_parc['Gain_€'].sum():,.0f} €
   • Gain moyen par parc : {meilleurs_par_parc['Gain_€'].mean():,.0f} €
   • Écart-type des gains : {meilleurs_par_parc['Gain_€'].std():,.0f} €

💡 RECOMMANDATIONS :
   1. Déployer le modèle probabiliste avec seuil {seuil_optimal:.1%} sur l'ensemble des parcs
   2. Pour les parcs où un benchmark est meilleur, utiliser un seuil personnalisé
   3. Surveiller l'évolution des pertes et ré-optimiser le seuil trimestriellement
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")



